# 2.0-feature-engineering

## Imports

In [1]:
import pandas as pd
from recruit_restaurant_visitor_forecasting.config.config import (
    AIR_AREA_COL,
    AIR_GENRE_COL,
    AIR_RESTAURANT_ID_COL,
    CALENDAR_DATE_COL,
    HPG_RESTAURANT_ID_COL,
    LATITUDE_COL,
    LONGITUDE_COL,
    VISIT_DATE_COL,
    VISITORS_COL,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
)
from recruit_restaurant_visitor_forecasting.config.features import (
    CITY_COL,
    DAY_OF_WEEK_COL,
    DAYS_OF_WEEK,
    OPEN_DATE_COL,
    RESERVE_AIR_COL,
    RESERVE_AIR_NBR_COL,
    RESERVE_HPG_COL,
    RESERVE_HPG_NBR_COL,
    TOTAL_RES_COL,
    TOTAL_RES_NBR_COL,
    VISITORS_NBR_COL,
    VISITORS_DOW,
    VISITORS_DOW_NBRS,
    RES_VISITORS_DIFF_COL,
    RES_VISITORS_DIFF_NBR_COL,
    GENRE_TE,
    AREA_TE,
    MEAN_PREF,
    MEDIAN_PREF,
    STD_PREF,
    DOW_WINDOW,
    RES_IMPOSSIBILITY_COL,
)
from recruit_restaurant_visitor_forecasting.config.preprocessing import EMPTY_DATES_RANGE, RES_MAX_OFFSET
from recruit_restaurant_visitor_forecasting.dataset import (
    prepare_datetime_columns, standardize_date
)
from recruit_restaurant_visitor_forecasting.features import (
    add_basic_stats,
    add_golden_week_flg,
    add_dow_cum_agg,
    add_holiday_columns,
    add_lags,
    add_last_month_visitors,
    add_nbrs_reserves,
    add_neighbors_stats,
    add_opened_recently_flg,
    add_reserves_difference,
    add_sum_of_reserves,
    add_time_based_target_encoding,
    add_total_nbr_reservations,
    add_total_reservations,
    drop_first_month,
    add_open_usually_discr_rolling,
    add_reservation_impossibility,
    add_dow_rol_agg,
    remove_repetitions,
    add_week_after_gw,
    add_hol_to_workday,
    add_rolling_res
)

2026-05-24 20:25:22.323 | INFO     | recruit_restaurant_visitor_forecasting.config.config:<module>:12 - PROJ_ROOT path is: D:\mentoring_program


In [2]:
air_visit_df = pd.read_csv(INTERIM_DATA_DIR / 'air_visit.csv')
air_reserve_df = pd.read_csv(INTERIM_DATA_DIR / 'air_reserve.csv')
hpg_reserve_df = pd.read_csv(INTERIM_DATA_DIR / 'hpg_reserve.csv', dtype={AIR_RESTAURANT_ID_COL: str})
future_df = pd.read_csv(INTERIM_DATA_DIR / 'sample_submission.csv')
air_store_df = pd.read_csv(INTERIM_DATA_DIR / 'air_store_info.csv')
hpg_store_df = pd.read_csv(INTERIM_DATA_DIR / 'hpg_store_info.csv')
date_info_df = pd.read_csv(RAW_DATA_DIR / 'date_info.csv')
store_rel_df = pd.read_csv(RAW_DATA_DIR / 'store_id_relation.csv')

In [3]:
prepare_datetime_columns(air_reserve_df)
prepare_datetime_columns(hpg_reserve_df)

standardize_date(date_info_df, CALENDAR_DATE_COL)
standardize_date(air_visit_df, VISIT_DATE_COL)
standardize_date(future_df, VISIT_DATE_COL)

## Features

### Air & hpg stores

In [4]:
air_store_df = air_store_df.drop([AIR_AREA_COL, LATITUDE_COL, LONGITUDE_COL], axis=1)

### Reserve dataframes

For further work, it is necessary to know the total number of reservations in the restaurant per day.

In [5]:
hpg_reserve_df = remove_repetitions(air_reserve_df, hpg_reserve_df)

In [6]:
air_res_sum = add_sum_of_reserves(air_reserve_df, RESERVE_AIR_COL, max_res_diff=RES_MAX_OFFSET)
air_res_sum = air_res_sum.merge(
    air_store_df[[AIR_RESTAURANT_ID_COL, CITY_COL]], on=AIR_RESTAURANT_ID_COL
)
air_res_sum = add_nbrs_reserves(air_res_sum, RESERVE_AIR_COL, RESERVE_AIR_NBR_COL, max_res_diff=RES_MAX_OFFSET)
air_res_sum.head()

,air_store_id,visit_date,air_reserves,air_reserves_0,air_reserves_1,air_reserves_2,air_reserves_3,air_reserves_4,air_reserves_5,air_reserves_6,...,air_reserves_nbrs_29,air_reserves_nbrs_30,air_reserves_nbrs_31,air_reserves_nbrs_32,air_reserves_nbrs_33,air_reserves_nbrs_34,air_reserves_nbrs_35,air_reserves_nbrs_36,air_reserves_nbrs_37,air_reserves_nbrs_38
0,air_00a91d42b08b08d9,2016-10-31,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,air_00a91d42b08b08d9,2016-12-05,9,9.0,9.0,9.0,9.0,0.0,0.0,0.0,...,0.813953,0.813953,0.813953,0.813953,0.813953,0.813953,0.720930,0.720930,0.720930,0.720930
2,air_00a91d42b08b08d9,2016-12-14,18,18.0,18.0,18.0,18.0,18.0,18.0,0.0,...,0.706897,0.327586,0.327586,0.275862,0.275862,0.275862,0.155172,0.155172,0.155172,0.155172
3,air_00a91d42b08b08d9,2016-12-17,2,2.0,2.0,2.0,2.0,2.0,2.0,0.0,...,3.771429,3.771429,3.528571,1.942857,1.942857,1.857143,1.857143,1.857143,1.857143,1.828571
4,air_00a91d42b08b08d9,2016-12-20,4,4.0,4.0,0.0,0.0,0.0,0.0,0.0,...,1.684211,1.684211,1.684211,1.684211,1.684211,1.385965,1.315789,1.315789,1.315789,1.315789


In [7]:
air_res_sum[RESERVE_AIR_NBR_COL].isna().sum()

np.int64(0)

However, air_reserve dataframe still has a large number of gaps.

In [8]:
hpg_res_sum = add_sum_of_reserves(
    hpg_reserve_df, RESERVE_HPG_COL, HPG_RESTAURANT_ID_COL, max_res_diff=RES_MAX_OFFSET
)
hpg_res_sum = hpg_res_sum.merge(
    hpg_store_df[[HPG_RESTAURANT_ID_COL, CITY_COL]], on=HPG_RESTAURANT_ID_COL
)
hpg_res_sum = add_nbrs_reserves(
    hpg_res_sum, RESERVE_HPG_COL, RESERVE_HPG_NBR_COL, max_res_diff=RES_MAX_OFFSET
)
hpg_res_sum.head()

,hpg_store_id,visit_date,hpg_reserves,hpg_reserves_0,hpg_reserves_1,hpg_reserves_2,hpg_reserves_3,hpg_reserves_4,hpg_reserves_5,hpg_reserves_6,...,hpg_reserves_nbrs_29,hpg_reserves_nbrs_30,hpg_reserves_nbrs_31,hpg_reserves_nbrs_32,hpg_reserves_nbrs_33,hpg_reserves_nbrs_34,hpg_reserves_nbrs_35,hpg_reserves_nbrs_36,hpg_reserves_nbrs_37,hpg_reserves_nbrs_38
0,hpg_001ce40a1f873e4f,2016-01-13,4,4.0,4.0,4.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,hpg_001ce40a1f873e4f,2016-01-27,7,7.0,7.0,7.0,7.0,7.0,7.0,7.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,hpg_001ce40a1f873e4f,2016-02-13,2,2.0,2.0,2.0,2.0,0.0,0.0,0.0,...,0.166667,0.166667,0.148148,0.148148,0.148148,0.129630,0.055556,0.000000,0.000000,0.000000
3,hpg_001ce40a1f873e4f,2016-02-27,8,8.0,8.0,8.0,8.0,8.0,8.0,8.0,...,1.096491,0.929825,0.929825,0.912281,0.912281,0.877193,0.877193,0.877193,0.877193,0.877193
4,hpg_001ce40a1f873e4f,2016-03-16,2,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.254237,0.254237,0.254237,0.254237,0.254237,0.254237,0.254237,0.254237,0.000000,0.000000


In [9]:
hpg_res_sum[RESERVE_HPG_NBR_COL].isna().sum()

np.int64(11535)

In [10]:
hpg_res_sum_mapped = hpg_res_sum.merge(store_rel_df, on=HPG_RESTAURANT_ID_COL)

### Date info

It is necessary to add a feature for the distance to the nearest holiday.

In [11]:
date_info_df = add_holiday_columns(date_info_df, CALENDAR_DATE_COL)
date_info_df = add_hol_to_workday(date_info_df)

It is also necessary to designate Golden Week, since not all days of this week are holidays.

In [12]:
date_info_df = add_golden_week_flg(date_info_df, [2016, 2017], CALENDAR_DATE_COL)
date_info_df = add_week_after_gw(date_info_df, CALENDAR_DATE_COL)

Later we will need to match the weekday of a selected date with the restaurant’s open/closed flag for that weekday, so it makes sense to rename the days.

In [13]:
FULL_WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
mapping = dict(zip(FULL_WEEKDAYS, DAYS_OF_WEEK))
date_info_df[DAY_OF_WEEK_COL] = date_info_df[DAY_OF_WEEK_COL].replace(mapping)

### Week after GW

### Air visit

#### Start dates

We can consider the restaurant's opening date. Since simply counting the number of days since opening may not be sufficient due to the increase in days over time, it's best to flag the restaurant's opening as occurring within the last six months. A **potential issue**: some restaurants either planned to open earlier than their minimum opening date but didn't, or didn't report visitors for earlier dates. This is indicated by the fact that air_reserve dataframe has reservations for earlier dates.

In [14]:
air_open_dates = (
    air_visit_df.groupby(AIR_RESTAURANT_ID_COL)[VISIT_DATE_COL]
    .min()
    .rename(OPEN_DATE_COL)
)

In [15]:
air_visit_df = add_opened_recently_flg(
    air_visit_df, air_open_dates, VISIT_DATE_COL, AIR_RESTAURANT_ID_COL
)
future_df = add_opened_recently_flg(
    future_df, air_open_dates, VISIT_DATE_COL, AIR_RESTAURANT_ID_COL
)

In [16]:
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1


In [17]:
future_df.head()

,id,visitors,air_store_id,visit_date,open_date,opened_recently
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0


#### Open dates

In [18]:
air_visit_df = (
    air_visit_df.merge(date_info_df, left_on=VISIT_DATE_COL, right_on=CALENDAR_DATE_COL)
    .merge(air_store_df, on=AIR_RESTAURANT_ID_COL)
    .drop(CALENDAR_DATE_COL, axis=1)
)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,golden_week_flg,week_after_gw_flg,air_genre_name,city
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,0,0,Italian/French,Tōkyō-to
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,0,0,Italian/French,Tōkyō-to
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,0,0,Italian/French,Tōkyō-to
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,0,0,Italian/French,Tōkyō-to
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,0,0,Italian/French,Tōkyō-to


In [19]:
air_visit_df = add_open_usually_discr_rolling(air_visit_df, 0.5)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,golden_week_flg,week_after_gw_flg,air_genre_name,city,open_usually
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,0,0,Italian/French,Tōkyō-to,NaN
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,0,0,Italian/French,Tōkyō-to,NaN
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,0,0,Italian/French,Tōkyō-to,NaN
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,0,0,Italian/French,Tōkyō-to,NaN
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,0,0,Italian/French,Tōkyō-to,NaN


Open_usually feature contains the probability that a restaurant will be open on a given day. It depends on the holiday feature, the percentage of non-zeros on holidays, and on individual days of the week.

In [20]:
future_df = (
    future_df.merge(date_info_df, left_on=VISIT_DATE_COL, right_on=CALENDAR_DATE_COL)
    .merge(air_store_df, on=AIR_RESTAURANT_ID_COL)
    .drop(CALENDAR_DATE_COL, axis=1)
)

#### Mean visitors by air city.

It is necessary to make smoothed target encoding, while the average value will be used for new areas. In this case, it is worth considering only days when the number of visitors is not zero, so that only open restaurants are taken into account.

In [21]:
air_visit_df, future_df = add_time_based_target_encoding(
    air_visit_df, future_df, CITY_COL, VISITORS_COL, AREA_TE
)

In [22]:
air_visit_df[
    [AIR_RESTAURANT_ID_COL, VISITORS_COL, AREA_TE, CITY_COL, VISIT_DATE_COL]
].head()

,air_store_id,visitors,air_city_te,city,visit_date
0,air_00a91d42b08b08d9,35,20.225068,Tōkyō-to,2016-07-01
1,air_00a91d42b08b08d9,9,20.301080,Tōkyō-to,2016-07-02
2,air_00a91d42b08b08d9,0,20.363266,Tōkyō-to,2016-07-03
3,air_00a91d42b08b08d9,20,20.373680,Tōkyō-to,2016-07-04
4,air_00a91d42b08b08d9,25,20.320037,Tōkyō-to,2016-07-05


In [23]:
future_df[[AIR_RESTAURANT_ID_COL, AREA_TE, CITY_COL, VISIT_DATE_COL]].head()

,air_store_id,air_city_te,city,visit_date
0,air_00a91d42b08b08d9,19.052358,Tōkyō-to,2017-04-23
1,air_00a91d42b08b08d9,19.052358,Tōkyō-to,2017-04-24
2,air_00a91d42b08b08d9,19.052358,Tōkyō-to,2017-04-25
3,air_00a91d42b08b08d9,19.052358,Tōkyō-to,2017-04-26
4,air_00a91d42b08b08d9,19.052358,Tōkyō-to,2017-04-27


#### Mean visitors by air genre

It is necessary to make smoothed target encoding, while the average value will be used for new genres.

In [24]:
air_visit_df, future_df = add_time_based_target_encoding(
    air_visit_df, future_df, AIR_GENRE_COL, VISITORS_COL, GENRE_TE
)

In [25]:
air_visit_df[
    [AIR_RESTAURANT_ID_COL, VISITORS_COL, GENRE_TE, AIR_GENRE_COL, VISIT_DATE_COL]
].head()

,air_store_id,visitors,air_genre_te,air_genre_name,visit_date
0,air_00a91d42b08b08d9,35,21.216062,Italian/French,2016-07-01
1,air_00a91d42b08b08d9,9,21.279619,Italian/French,2016-07-02
2,air_00a91d42b08b08d9,0,21.352360,Italian/French,2016-07-03
3,air_00a91d42b08b08d9,20,21.376200,Italian/French,2016-07-04
4,air_00a91d42b08b08d9,25,21.334206,Italian/French,2016-07-05


In [26]:
future_df[[AIR_RESTAURANT_ID_COL, GENRE_TE, AIR_GENRE_COL, VISIT_DATE_COL]].head()

,air_store_id,air_genre_te,air_genre_name,visit_date
0,air_00a91d42b08b08d9,20.969311,Italian/French,2017-04-23
1,air_00a91d42b08b08d9,20.969311,Italian/French,2017-04-24
2,air_00a91d42b08b08d9,20.969311,Italian/French,2017-04-25
3,air_00a91d42b08b08d9,20.969311,Italian/French,2017-04-26
4,air_00a91d42b08b08d9,20.969311,Italian/French,2017-04-27


#### Total reserved visitors

Total reserved visitors (from air_reserve and hpg_reserve) on this day - for this restaurant/for neighbors.

By neighbors, we designate restaurants that are located in the same city.

In [27]:
air_visit_df = add_reservation_impossibility(air_visit_df, air_res_sum, hpg_res_sum_mapped)
air_visit_df = add_total_reservations(
    air_visit_df, air_res_sum, hpg_res_sum_mapped, CITY_COL, RES_MAX_OFFSET
)
air_visit_df = add_total_nbr_reservations(
    air_visit_df, air_res_sum, hpg_res_sum_mapped, RES_MAX_OFFSET
)
air_visit_df.loc[air_visit_df[VISIT_DATE_COL].isin(pd.to_datetime(EMPTY_DATES_RANGE)), RES_IMPOSSIBILITY_COL] = 1
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,golden_week_flg,...,total_reservations_nbrs_30,total_reservations_nbrs_31,total_reservations_nbrs_32,total_reservations_nbrs_33,total_reservations_nbrs_34,total_reservations_nbrs_35,total_reservations_nbrs_36,total_reservations_nbrs_37,total_reservations_nbrs_38,total_reservations_nbrs_39
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,0,...,0.255575,0.248714,0.238422,0.222985,0.222985,0.222985,0.171527,0.171527,0.150943,0.150943
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,0,...,1.474468,1.470213,1.457447,1.427660,1.062411,1.062411,1.062411,0.998582,0.900709,0.890071
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,0,...,0.534221,0.534221,0.519011,0.519011,0.519011,0.519011,0.519011,0.519011,0.519011,0.519011
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,0,...,0.010695,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,0,...,0.284047,0.284047,0.284047,0.284047,0.190661,0.171206,0.171206,0.171206,0.035019,0.035019


In [29]:
future_df = add_reservation_impossibility(future_df, air_res_sum, hpg_res_sum_mapped)
future_df = add_total_reservations(
    future_df, air_res_sum, hpg_res_sum_mapped, CITY_COL, RES_MAX_OFFSET
)
future_df = add_total_nbr_reservations(
    future_df, air_res_sum, hpg_res_sum, RES_MAX_OFFSET
)
future_df.head()

,id,visitors,air_store_id,visit_date,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,...,total_reservations_nbrs_30,total_reservations_nbrs_31,total_reservations_nbrs_32,total_reservations_nbrs_33,total_reservations_nbrs_34,total_reservations_nbrs_35,total_reservations_nbrs_36,total_reservations_nbrs_37,total_reservations_nbrs_38,total_reservations_nbrs_39
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0,Sun,0,-6,0,...,0.482464,0.468492,0.406615,0.402623,0.362703,0.322783,0.322783,0.322783,0.322783,0.302823
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0,Mon,0,-5,0,...,0.514360,0.514360,0.454308,0.454308,0.454308,0.206266,0.201044,0.190601,0.190601,0.190601
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0,Tue,0,-4,0,...,0.175166,0.170732,0.130820,0.130820,0.104213,0.104213,0.104213,0.099778,0.084257,0.084257
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0,Wed,0,-3,0,...,0.668187,0.620976,0.618830,0.618830,0.431330,0.324034,0.324034,0.259657,0.259657,0.259657
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0,Th,0,-2,0,...,0.408333,0.237255,0.237255,0.229902,0.205392,0.205392,0.205392,0.205392,0.205392,0.205392


#### Rolling mean/median/std of visitors

If open_usually == 0 and visitors == 0, the value is replaced with NaN. Otherwise, zero is included in the aggregation. Later, we need to take care of the initial rows of the window, because right now they contain non-NaN values, even though conceptually they should be NaN.

In [31]:
aggs = [
    ("mean", {}),
    ("median", {}),
    ("std", {"ddof": 0}),
    ("max", {}),
    ("min", {}),
]
air_visit_df = add_basic_stats(air_visit_df, VISITORS_COL, AIR_RESTAURANT_ID_COL, aggs=aggs)
air_visit_df = add_neighbors_stats(air_visit_df, VISITORS_COL, CITY_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,golden_week_flg,...,visitors_nbrs,visitors_nbrs_mean_7,visitors_nbrs_median_7,visitors_nbrs_std_7,visitors_nbrs_mean_14,visitors_nbrs_median_14,visitors_nbrs_std_14,visitors_nbrs_mean_28,visitors_nbrs_median_28,visitors_nbrs_std_28
0,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,0,...,24.995192,20.693662,20.076923,2.853880,20.400778,19.589744,3.123405,19.974622,18.835153,3.233809
1,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,0,...,23.384615,20.892270,20.076923,3.088318,20.419716,19.589744,3.150285,20.079609,18.835153,3.345454
2,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,0,...,16.738095,20.818555,20.076923,3.021057,20.329176,19.589744,3.043756,20.103836,18.835153,3.366777
3,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,0,...,14.350254,19.809475,19.102564,3.035765,19.954506,18.605769,3.138428,19.921964,18.408871,3.405908
4,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,0,...,17.213429,19.508189,19.102564,3.432093,19.829204,18.605769,3.319438,19.888869,18.408871,3.455031


#### Visitors lag features

In [32]:
air_visit_df = add_lags(air_visit_df, AIR_RESTAURANT_ID_COL, VISITORS_COL)
air_visit_df = add_lags(air_visit_df, CITY_COL, VISITORS_NBR_COL, True)
air_visit_df.tail()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,golden_week_flg,...,visitors_nbrs_std_14,visitors_nbrs_mean_28,visitors_nbrs_median_28,visitors_nbrs_std_28,visitors_lag_1,visitors_lag_7,visitors_lag_28,visitors_nbrs_lag_1,visitors_nbrs_lag_7,visitors_nbrs_lag_28
254021,air_db4b38ebe7a7ceff,2017-04-22,19,2016-01-01,0,Sat,0,-7,0,0,...,4.150945,19.809507,19.499812,4.109034,17.0,28.0,25.0,22.243243,25.465753,28.555556
255091,air_dc0e080ba0a5e5af,2017-04-22,10,2016-07-01,0,Sat,0,-7,0,0,...,4.150945,19.809507,19.499812,4.109034,10.0,15.0,22.0,22.243243,25.465753,28.555556
257112,air_dea0655f96947922,2017-04-22,64,2016-01-02,0,Sat,0,-7,0,0,...,4.150945,19.809507,19.499812,4.109034,30.0,55.0,54.0,22.243243,25.465753,28.555556
276170,air_eda179770dfa9f91,2017-04-22,21,2016-07-01,0,Sat,0,-7,0,0,...,4.150945,19.809507,19.499812,4.109034,19.0,6.0,17.0,22.243243,25.465753,28.555556
280270,air_efef1e3daecce07e,2017-04-22,47,2016-01-05,0,Sat,0,-7,0,0,...,4.150945,19.809507,19.499812,4.109034,38.0,47.0,51.0,22.243243,25.465753,28.555556


#### Visitors on the same day last month


The feature has a drawback: if the previous month had 30 days and the current month has 31, then the feature for the 31st day will correspond to the 30th day. If there were 0 visitors, and the open_usually flag is 0, then 0 visitors are indicated.

In [33]:
air_visit_df = add_last_month_visitors(air_visit_df)
air_visit_df.tail()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,golden_week_flg,...,visitors_nbrs_mean_28,visitors_nbrs_median_28,visitors_nbrs_std_28,visitors_lag_1,visitors_lag_7,visitors_lag_28,visitors_nbrs_lag_1,visitors_nbrs_lag_7,visitors_nbrs_lag_28,visitors_last_month
296883,air_db4b38ebe7a7ceff,2017-04-22,19,2016-01-01,0,Sat,0,-7,0,0,...,19.809507,19.499812,4.109034,17.0,28.0,25.0,22.243243,25.465753,28.555556,1.0
296884,air_dc0e080ba0a5e5af,2017-04-22,10,2016-07-01,0,Sat,0,-7,0,0,...,19.809507,19.499812,4.109034,10.0,15.0,22.0,22.243243,25.465753,28.555556,6.0
296885,air_dea0655f96947922,2017-04-22,64,2016-01-02,0,Sat,0,-7,0,0,...,19.809507,19.499812,4.109034,30.0,55.0,54.0,22.243243,25.465753,28.555556,32.0
296886,air_eda179770dfa9f91,2017-04-22,21,2016-07-01,0,Sat,0,-7,0,0,...,19.809507,19.499812,4.109034,19.0,6.0,17.0,22.243243,25.465753,28.555556,13.0
296887,air_efef1e3daecce07e,2017-04-22,47,2016-01-05,0,Sat,0,-7,0,0,...,19.809507,19.499812,4.109034,38.0,47.0,51.0,22.243243,25.465753,28.555556,39.0


#### Historical day-of-week mean of visitors


In [34]:
aggs = [MEAN_PREF, MEDIAN_PREF, STD_PREF]
air_visit_df, future_df = add_dow_cum_agg(
    air_visit_df, VISITORS_COL, VISITORS_DOW, aggs, future_df
)
air_visit_df, future_df = add_dow_cum_agg(
    air_visit_df, VISITORS_NBR_COL, VISITORS_DOW_NBRS, aggs, future_df
)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,golden_week_flg,...,visitors_nbrs_lag_1,visitors_nbrs_lag_7,visitors_nbrs_lag_28,visitors_last_month,visitors_dow_mean,visitors_dow_median,visitors_dow_std,visitors_nbrs_dow_mean,visitors_nbrs_dow_median,visitors_nbrs_dow_std
0,air_05c325d315cc17f5,2016-01-01,29,2016-01-01,1,Fri,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,air_09a845d5b5944b01,2016-01-01,56,2016-01-01,1,Fri,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,air_298513175efdf261,2016-01-01,12,2016-01-01,1,Fri,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,air_60a7057184ec7ec7,2016-01-01,64,2016-01-01,1,Fri,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,air_87f9e1024b951f01,2016-01-01,17,2016-01-01,1,Fri,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
future_df.head()

,id,visitors,air_store_id,visit_date,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,...,total_reservations_nbrs_36,total_reservations_nbrs_37,total_reservations_nbrs_38,total_reservations_nbrs_39,visitors_dow_mean,visitors_dow_median,visitors_dow_std,visitors_nbrs_dow_mean,visitors_nbrs_dow_median,visitors_nbrs_dow_std
0,air_00a91d42b08b08d9_2017-04-23,0,air_00a91d42b08b08d9,2017-04-23,2016-07-01,0,Sun,0,-6,0,...,0.322783,0.322783,0.322783,0.302823,0.048780,0.0,0.312348,19.779251,20.449686,3.139300
1,air_00a91d42b08b08d9_2017-04-24,0,air_00a91d42b08b08d9,2017-04-24,2016-07-01,0,Mon,0,-5,0,...,0.201044,0.190601,0.190601,0.190601,18.707317,18.0,12.209103,14.523331,14.809917,2.306340
2,air_00a91d42b08b08d9_2017-04-25,0,air_00a91d42b08b08d9,2017-04-25,2016-07-01,0,Tue,0,-4,0,...,0.104213,0.099778,0.084257,0.084257,22.902439,24.0,10.261103,16.076911,16.439320,2.619608
3,air_00a91d42b08b08d9_2017-04-26,0,air_00a91d42b08b08d9,2017-04-26,2016-07-01,0,Wed,0,-3,0,...,0.324034,0.259657,0.259657,0.259657,27.024390,28.0,10.588881,18.034140,17.966981,2.427093
4,air_00a91d42b08b08d9_2017-04-27,0,air_00a91d42b08b08d9,2017-04-27,2016-07-01,0,Th,0,-2,0,...,0.205392,0.205392,0.205392,0.205392,26.756098,29.0,11.173139,17.754260,18.009456,2.189099


In [36]:
air_visit_df = add_dow_rol_agg(air_visit_df, VISITORS_COL, VISITORS_DOW, aggs, DOW_WINDOW)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,golden_week_flg,...,visitors_last_month,visitors_dow_mean,visitors_dow_median,visitors_dow_std,visitors_nbrs_dow_mean,visitors_nbrs_dow_median,visitors_nbrs_dow_std,visitors_dow_mean_4,visitors_dow_median_4,visitors_dow_std_4
0,air_05c325d315cc17f5,2016-01-01,29,2016-01-01,1,Fri,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,air_09a845d5b5944b01,2016-01-01,56,2016-01-01,1,Fri,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,air_298513175efdf261,2016-01-01,12,2016-01-01,1,Fri,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,air_60a7057184ec7ec7,2016-01-01,64,2016-01-01,1,Fri,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,air_87f9e1024b951f01,2016-01-01,17,2016-01-01,1,Fri,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Rolling reserve/visitors difference


In [37]:
air_visit_df = add_reserves_difference(
    air_visit_df, VISITORS_COL, TOTAL_RES_COL, RES_VISITORS_DIFF_COL
)
air_visit_df = add_reserves_difference(
    air_visit_df, VISITORS_NBR_COL, TOTAL_RES_NBR_COL, RES_VISITORS_DIFF_NBR_COL
)

In [38]:
aggs = [("mean", {})]
air_visit_df = add_basic_stats(
    air_visit_df, RES_VISITORS_DIFF_COL, AIR_RESTAURANT_ID_COL, aggs
)
air_visit_df = add_neighbors_stats(
    air_visit_df, RES_VISITORS_DIFF_NBR_COL, CITY_COL, aggs, False
)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,golden_week_flg,...,visitors_dow_median_4,visitors_dow_std_4,res_visitors_diff,res_visitors_diff_mean_7,res_visitors_diff_mean_14,res_visitors_diff_mean_28,nbr_res_visitors_diff,nbr_res_visitors_diff_mean_7,nbr_res_visitors_diff_mean_14,nbr_res_visitors_diff_mean_28
140971,air_00a91d42b08b08d9,2016-07-01,35,2016-07-01,1,Fri,0,-17,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,-6.642857,-6.733519,-7.305495,-7.708858
141388,air_00a91d42b08b08d9,2016-07-02,9,2016-07-01,1,Sat,0,-16,0,0,...,NaN,NaN,-35.0,NaN,NaN,NaN,-9.271350,-6.416572,-7.354003,-7.609692
141809,air_00a91d42b08b08d9,2016-07-03,0,2016-07-01,1,Sun,0,-15,0,0,...,NaN,NaN,-9.0,-35.000000,-35.000000,-35.000000,-11.066176,-7.004593,-7.255023,-7.697994
142231,air_00a91d42b08b08d9,2016-07-04,20,2016-07-01,1,Mon,0,-14,0,0,...,NaN,NaN,0.0,-22.000000,-22.000000,-22.000000,-7.177259,-8.361577,-7.231215,-7.868184
142654,air_00a91d42b08b08d9,2016-07-05,25,2016-07-01,1,Tue,0,-13,0,0,...,NaN,NaN,-20.0,-14.666667,-14.666667,-14.666667,-5.141698,-7.883773,-6.625714,-7.885627


## Saving

#### NaN dropping

In [39]:
air_visit_df.to_csv(PROCESSED_DATA_DIR / "air_visit.csv", index=False)
air_visit_df = drop_first_month(air_visit_df)
air_visit_df = air_visit_df.sort_values(VISIT_DATE_COL)
air_visit_df.head()

,air_store_id,visit_date,visitors,open_date,opened_recently,day_of_week,holiday_flg,days_from_holiday,holiday_to_workday_flg,golden_week_flg,...,visitors_dow_median_4,visitors_dow_std_4,res_visitors_diff,res_visitors_diff_mean_7,res_visitors_diff_mean_14,res_visitors_diff_mean_28,nbr_res_visitors_diff,nbr_res_visitors_diff_mean_7,nbr_res_visitors_diff_mean_14,nbr_res_visitors_diff_mean_28
116844,air_8d50c64692322dff,2016-02-01,0,2016-01-01,1,Mon,0,-10,0,0,...,4.5,10.862780,-5.0,-1.666667,-1.461538,-2.481481,-5.584158,-3.832741,-4.223893,-6.903372
74495,air_35c4732dcbfe31be,2016-02-01,0,2016-01-01,1,Mon,0,-10,0,0,...,8.5,2.160247,-18.0,-6.285714,-7.500000,-7.714286,-5.109063,-4.291707,-3.715279,-6.460045
270949,air_536043fcf1a4f8a4,2016-02-01,33,2016-01-01,1,Mon,0,-10,0,0,...,28.0,18.025445,-32.0,-25.714286,-26.928571,-26.357143,-14.426903,-6.081161,-7.088246,-9.822531
74496,air_39dccf7df20b1c6a,2016-02-01,26,2016-01-01,1,Mon,0,-10,0,0,...,26.0,6.500000,-38.0,-18.714286,-21.928571,-23.571429,-5.109063,-4.291707,-3.715279,-6.460045
116780,air_36bcf77d3382d36e,2016-02-01,17,2016-01-01,1,Mon,0,-10,0,0,...,20.5,19.200694,-56.0,-29.857143,-28.000000,-27.785714,-5.584158,-3.832741,-4.223893,-6.903372


In [40]:
air_visit_df = air_visit_df.drop(columns=[OPEN_DATE_COL])
labels = air_visit_df[VISITORS_COL].copy()
features = air_visit_df.drop(columns=[VISITORS_COL])
future_df = future_df.drop(columns=[OPEN_DATE_COL, "id", VISITORS_COL])
future_df[TOTAL_RES_NBR_COL] = future_df[TOTAL_RES_NBR_COL].fillna(0)
features.to_csv(PROCESSED_DATA_DIR / "features.csv", index=False)
labels.to_csv(PROCESSED_DATA_DIR / "labels.csv", index=False)
future_df.to_csv(PROCESSED_DATA_DIR / "test_features.csv", index=False)

## Conclusion

### Added features.

- Operating schedule - regular working days, extracted from the regular gaps in the data frame.
- Days to/from the nearest holiday (negative is days until, positive is days after).
- Holiday indicator (currently included in date_info).
- Separate indicators for Golden Week dates.

- Number of visitors on this day last month for this restaurant.

- Indicator of whether the restaurant has been open within the last 6 months.
- Days since last recorded visit for air_visit dataframe.

- Rolling mean/median/std of visitors over the past week, month - for this restaurant/for neighbors.
- Historical day-of-week mean up to (but not including) the current day - for this restaurant/for neighbors.
- Rolling reserve/visitors difference over the past 7 / 28 days - for this restaurant/for neighbors.
- Lag 1, 7, 28 of the number of visitors - for this restaurant/for neighbors.

- Smoothed target encoding of visitors by air_area_name.
- Smoothed target encoding of visitors by air_genre_name.

- Total reserved visitors (from air_reserve and hpg_reserve) on this day - for this restaurant/for neighbors.

### Features planned for addition.

- Days since last recorded visit for sample_submission.

### Possible features.

- Daily temperature - try to get the weather forecast.
- Precipitation probability.
- Hpg genre.

Decomposition features:
- Trend.
- Trend difference for last month.
- Seasonal.
- Residual mean for last month.